<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/thirdWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 - RNN, LSTM, Attention, Transformers

Week 2 me MLP tak pahunche the (fixed context size). Problem: context badhao to params bhi badhte hai. **RNN** isko sequence me solve karta hai (ek hidden state ko aage carry karo).

Is week me sab kuch **scratch se** (PyTorch layers use karke, but cell khud likha):
1. RNN cell manually -> char level naam model
2. LSTM cell manually (gates samajhne ke liye)
3. self-**attention** (single head)
4. multi-head + **transformer** block -> chhota GPT

Same names.txt dataset use kar raha hu taaki compare kar saku.

In [ ]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt', 'names.txt')
words = open('names.txt').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('vocab size:', vocab_size, '| names:', len(words))

## Dataset (padded sequences)

Har naam ko `.` + naam + `.` banate hai. Input = pura naam except last char, target = ek step aage shift. Batch banane ke liye sabko max length tak `.` (index 0) se pad kar dete hai.

In [ ]:
max_len = max(len(w) for w in words) + 1   # ek extra '.' ke liye


def encode_name(w):
    seq = [stoi['.']] + [stoi[c] for c in w] + [stoi['.']]
    x = seq[:-1]
    y = seq[1:]
    # pad karo max_len tak
    pad = max_len + 1 - len(seq)
    x = x + [0] * pad
    y = y + [0] * pad
    return x[:max_len], y[:max_len]


X, Y = [], []
for w in words:
    x, y = encode_name(w)
    X.append(x)
    Y.append(y)
X = torch.tensor(X)
Y = torch.tensor(Y)
print('X:', X.shape, '| Y:', Y.shape)

n1 = int(0.9 * len(X))
Xtr, Ytr = X[:n1].to(device), Y[:n1].to(device)
Xdev, Ydev = X[n1:].to(device), Y[n1:].to(device)

## 1. RNN cell (scratch)

Formula: `h_t = tanh(Wxh * x_t + Whh * h_{t-1} + b)`. Bas ek hidden state jo har step pe update hota hai. Do `nn.Linear` use kiye - ek input ke liye ek hidden ke liye.

In [ ]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.Wxh = nn.Linear(input_size, hidden_size)   # input -> hidden
        self.Whh = nn.Linear(hidden_size, hidden_size)  # hidden -> hidden

    def forward(self, x, h):
        return torch.tanh(self.Wxh(x) + self.Whh(h))


class CharRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cell = RNNCell(emb_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        # x: (batch, seq_len)
        B, T = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        e = self.emb(x)                       # (B, T, emb)
        outs = []
        for t in range(T):                    # time steps pe loop -> yahi RNN ka dil hai
            h = self.cell(e[:, t], h)
            outs.append(self.fc(h))
        return torch.stack(outs, dim=1)       # (B, T, vocab)

In [ ]:
def train_char_model(model, steps=3000, batch_size=64, lr=0.01):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), device=device)
        xb, yb = Xtr[ix], Ytr[ix]
        logits = model(xb)                    # (B, T, vocab)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if i % 500 == 0:
            print(f'step {i:5d} | loss {loss.item():.4f}')
    return model


rnn = CharRNN(vocab_size, emb_dim=24, hidden_size=128)
rnn = train_char_model(rnn)

In [ ]:
@torch.no_grad()
def sample_rnn_like(model, n=10):
    """RNN/LSTM dono ke liye ek-ek char generate karke naam banata hai."""
    model.eval()
    for _ in range(n):
        ix = 0
        out = []
        h = torch.zeros(1, model.hidden_size, device=device)
        c = torch.zeros(1, model.hidden_size, device=device)   # LSTM ke liye
        for _ in range(max_len):
            e = model.emb(torch.tensor([[ix]], device=device))[:, 0]
            if isinstance(model.cell, LSTMCell):
                h, c = model.cell(e, (h, c))
            else:
                h = model.cell(e, h)
            logits = model.fc(h)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, 1).item()
            if ix == 0:
                break
            out.append(itos[ix])
        print(''.join(out))
    model.train()

## 2. LSTM cell (scratch)

Plain RNN me long dependencies yaad nahi rehti (vanishing gradient). LSTM 3 gates deta hai:
- **forget gate** (f): pichli memory kitni bhoolni hai
- **input gate** (i): nayi info kitni daalni hai
- **output gate** (o): memory se kitna bahar dikhana hai

Ek `cell state` c hota hai jo lambi memory ki tarah kaam karta hai. Maine 4 gates ek saath compute kiye phir `chunk` se tod diya (fast + clean).

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # ek hi linear me 4 gates (i, f, g, o) -> baad me chunk
        self.i2h = nn.Linear(input_size, 4 * hidden_size)
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x, state):
        h, c = state
        gates = self.i2h(x) + self.h2h(h)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)      # input gate
        f = torch.sigmoid(f)      # forget gate
        g = torch.tanh(g)         # candidate memory
        o = torch.sigmoid(o)      # output gate
        c = f * c + i * g         # nayi cell state
        h = o * torch.tanh(c)     # nayi hidden state
        return h, c


class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cell = LSTMCell(emb_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        B, T = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        c = torch.zeros(B, self.hidden_size, device=x.device)
        e = self.emb(x)
        outs = []
        for t in range(T):
            h, c = self.cell(e[:, t], (h, c))
            outs.append(self.fc(h))
        return torch.stack(outs, dim=1)


lstm = CharLSTM(vocab_size, emb_dim=24, hidden_size=128)
lstm = train_char_model(lstm)
print('\nLSTM se naam:')
sample_rnn_like(lstm, n=10)

## 3. Self-Attention (single head)

RNN sequential hai (ek time me ek step). Attention idea: har token baaki sab (pichle) tokens ko dekhe aur decide kare kis pe kitna dhyan dena hai.

- **query** (q): main kya dhundh raha hu
- **key** (k): mere paas kya hai
- **value** (v): agar match hua to kya doonga

`wei = q @ k^T / sqrt(d)` -> lower-triangular mask (future dekhna mana) -> softmax -> `wei @ v`. Ye makemore/nanoGPT wala exact setup hai.

In [ ]:
class Head(nn.Module):
    """ek self-attention head (causal / masked)."""

    def __init__(self, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # future band
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

## 4. Multi-head attention + Transformer block -> chhota GPT

Kai heads parallel me alag alag cheez pe dhyan dete hai, phir concat. Uske baad feed-forward, aur residual + layernorm (jaise "Attention is all you need" me). Ye sab milke ek **transformer block** banta hai.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(n_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """transformer block: attention + feedforward, dono me residual + layernorm."""

    def __init__(self, n_embd, n_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, n_embd, head_size, block_size, dropout)
        self.ff = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual connection
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)   # position bhi seekhte hai
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_heads, block_size, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
# transformer ke liye ek continuous stream banate hai (naam '.' se separate)
block_size = 16
data = []
for w in words:
    data += [stoi[c] for c in w] + [stoi['.']]
data = torch.tensor(data, dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_batch(split, batch_size=64):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


gpt = MiniGPT(vocab_size, n_embd=64, n_heads=4, n_layers=3, block_size=block_size).to(device)
print('GPT params:', sum(p.numel() for p in gpt.parameters()))
opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)

for step in range(3000):
    xb, yb = get_batch('train')
    logits = gpt(xb)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f'step {step:5d} | loss {loss.item():.4f}')

In [ ]:
# GPT se naam generate karo
start = torch.tensor([[stoi['.']]], device=device)
out = gpt.generate(start, max_new_tokens=200)[0].tolist()

text = ''.join(itos[i] for i in out)
names = [nm for nm in text.split('.') if nm]
print('generated names:')
for nm in names[:20]:
    print(' ', nm)

## Week 3 summary (mera notes)

- **RNN**: ek hidden state carry karo, har step update -> sequence handle ho gaya, but lambi memory weak
- **LSTM**: gates (forget/input/output) + cell state -> lambi dependency better yaad
- **attention**: q,k,v se har token relevant tokens pe dhyan de sakta hai (parallel, sequential nahi)
- **transformer**: multi-head attention + feedforward + residual + layernorm -> chhota GPT bana liya

Ye sab milke final project (seq2seq **English -> German translator**) ki base ban gayi. Encoder-decoder wahi LSTM/attention ideas pe khada hai.